# Analsis of First Move Win Bias

In our current 1v1 agent matchups, there's clearly a bias towards the player who moves first. We can try to compensate for this by rotating seats per-map (run the same number of games over the same maps, but with the seats traded), and I recommend doing this.

But this is merely moving around this bias, not alleviating it. If there is an 80% win rate bias for the player moving first, we'll need a huge number of games to actually measure differences in "skill" between agents. With agents like Expectiminimax, which have tractability problems, we almost certainly will not be able to do this, therefore we should make some effort to first quantify and then alleviate First Player Move Bias.

## Quantify First Move Win Bias

This first experiment tries to quantify the first move win bias by just playing N games of the Greedy agent versus itself.

If there were no bias, you would think that for enough games we'd see the rate at which the first player wins move towards 50%. This experiment tries to quantify that by running an increasing number of experiments to see what first player win rate this gravitates towards.

So we do the following:

* Run an increasing number of games of greedy vs greed. Capture "percentage of first mover wins".
* Re-run each set N times (a small number to maintain tractabiity).
* We then can form a basic mean and standard deviation for the "first player win rate" (FPWR).
* Plot the results. We hopefully see them gravitate towards a known value.

In [ ]:
from nb_setup import risk_ai_game
import matplotlib.pyplot as plt
import random
import statistics

# run batches of games of increasing size.
RUN_COUNTS = [20, 40, 80, 100, 200, 400, 600, 800]
# and for every batch, run N times.
# I'm keeping this small to maintain tractability.
NUM_REPEATS = 4

# We regenerate random seeds every experiment from this master seed.
master_rng = random.Random(12345)

# data to collect.
mean_rates = []
std_rates = []

for num_games in RUN_COUNTS:
    batch_rates = []

    for _ in range(NUM_REPEATS):
        first_mover_wins = 0

        # each experiment has a dedicated set of seeds (no overlap of seeds between experiments).
        seeds = [master_rng.randrange(1, 10**9) for _ in range(num_games)]

        for seed in seeds:
            winner = risk_ai_game.run_game(
                risk_ai_game.RiskAIGameOptions(
                    agents=[
                        risk_ai_game.GreedyAgent(0, name="Greedy-P0"),
                        risk_ai_game.GreedyAgent(1, name="Greedy-P1"),
                    ],
                    max_turns=500,
                    verbose=False,
                    random_seed=seed,
                )
            )
            if winner == 0:
                first_mover_wins += 1

        batch_rates.append(100.0 * first_mover_wins / num_games)

    # form statistics over the N runs.
    mean_rates.append(statistics.mean(batch_rates))
    std_rates.append(statistics.pstdev(batch_rates))
    print(
        f"{num_games} games: mean={mean_rates[-1]:.2f}%, "
        f"std={std_rates[-1]:.2f}%"
    )

# Plot.
plt.figure(figsize=(8, 4))
plt.errorbar(RUN_COUNTS, mean_rates, yerr=std_rates, marker="o", capsize=4)
plt.xlabel("Games per experiment")
plt.ylabel("First player win rate (%)")
plt.title("Greedy vs Greedy: at number of games.")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Alleviating First Player Move Bias with Initial Armies

We now will try to alleviate the FPMB by increasing the number of armies present in territories at the start of the game, varying the number of armies to see where the FPMB is at a minimum.

In [ ]:
from nb_setup import risk_ai_game
import matplotlib.pyplot as plt
import random
import statistics

GAMES_PER_BATCH = 200
NUM_BATCHES = 3
INITIAL_ARMIES_PER_TERRITORY = range(1, 8)  # 1 .. 7

# a helper to initialize a random board but with N initial armies per territory.
def make_initial_board_setup(armies_per_territory: int):
    def setup(game: risk_ai_game.GameState) -> None:
        territories = game.board.all_territories()
        random.shuffle(territories)
        for i, t in enumerate(territories):
            t.owner = i % game.num_players
            t.armies = armies_per_territory
        game.armies_to_deploy = game.get_reinforcements(game.current_player)

    return setup

# collected data to plot.
armies_levels = []
mean_fpwr = []
std_fpwr = []

# run our experiments.
for armies in INITIAL_ARMIES_PER_TERRITORY:
    setup_board = make_initial_board_setup(armies)
    batch_rates = []

    for batch in range(NUM_BATCHES):
        first_mover_wins = 0
        for g in range(GAMES_PER_BATCH):
            seed = 1_000_000 * armies + 10_000 * batch + g
            winner = risk_ai_game.run_game(
                risk_ai_game.RiskAIGameOptions(
                    agents=[
                        risk_ai_game.GreedyAgent(0, name="Greedy-P0"),
                        risk_ai_game.GreedyAgent(1, name="Greedy-P1"),
                    ],
                    max_turns=500,
                    verbose=False,
                    random_seed=seed,
                    initial_board_setup=setup_board,
                )
            )
            if winner == 0:
                first_mover_wins += 1
        batch_rates.append(100.0 * first_mover_wins / GAMES_PER_BATCH)

    armies_levels.append(armies)
    mean_fpwr.append(statistics.mean(batch_rates))
    std_fpwr.append(statistics.pstdev(batch_rates) if NUM_BATCHES > 1 else 0.0)

    print(
        f"Initial armies/territory={armies}: "
        f"batch FPWRs={[round(r, 2) for r in batch_rates]}%, "
        f"mean={mean_fpwr[-1]:.2f}%, std={std_fpwr[-1]:.2f}%"
    )

plt.figure(figsize=(8, 4))
plt.errorbar(armies_levels, mean_fpwr, yerr=std_fpwr, marker="o", capsize=4)
plt.xlabel("Initial armies per territory")
plt.ylabel("First player win rate (%)")
plt.title("Greedy vs Greedy: FPWR vs starting armies (200 games with 3 batches each)")
plt.xticks(list(INITIAL_ARMIES_PER_TERRITORY))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()